# Genie: One, Agents, Ontology and Code

Genie is the Databricks family of AI experiences for asking questions of governed data in natural language. Data teams curate Genie Agents (formerly Genie spaces) with Unity Catalog datasets, example SQL, business semantics and instructions; Genie One gives business users one chat interface across that context; Genie Ontology supplies modelled and learned business meaning; Genie Code assists developers inside the workspace. Everything respects Unity Catalog permissions.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Genie: One, Agents, Ontology and Code](https://www.datapreplab.com/hub#topic/genie) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** curate a small Genie Agent and measure the effect of an instruction.

**Part A: governed data.** Create and comment two tables:

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.genie_customers (customer_id INT COMMENT 'Customer key', segment STRING COMMENT 'Enterprise or SMB')
COMMENT 'One row per active customer';

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.genie_orders (order_id INT, customer_id INT, order_date DATE, amount DECIMAL(10,2) COMMENT 'Order value in GBP')
COMMENT 'One row per order; amount in GBP';

In [ ]:
%sql
INSERT INTO workspace.default.genie_customers VALUES (1, 'Enterprise'), (2, 'SMB');

In [ ]:
%sql
INSERT INTO workspace.default.genie_orders VALUES
  (101, 1, DATE'2026-01-15', 500.00), (102, 2, DATE'2026-02-10', 120.00), (103, 1, DATE'2026-03-05', 300.00),
  (104, 2, DATE'2026-04-20', 80.00),  (105, 1, DATE'2026-05-12', 250.00);

**Part B: the agent.** Create a Genie Agent with both tables. Ask "What was revenue in Q1?" and record the SQL it writes.

**Part C: teach and test.** Predict how the answer changes, then:

1. **Instruction.** In the agent's instructions, add: "Our fiscal year starts on 1 February; Q1 is February to April."
2. **Example SQL.** Add this query with the title "Revenue by fiscal quarter". Shifting each date back one month makes February the first month of a fiscal year, so the built-in `year()` and `quarter()` give fiscal years and quarters:

```text
SELECT year(add_months(order_date, -1))    AS fiscal_year,
       quarter(add_months(order_date, -1)) AS fiscal_quarter,
       sum(amount)                         AS revenue
FROM workspace.default.genie_orders
GROUP BY ALL
ORDER BY fiscal_year, fiscal_quarter
```

A trap to avoid: `date_trunc('quarter', order_date)` groups by **calendar** quarters, so February and March land in one quarter and April in the next. Used as a "fiscal" example, it teaches the agent the opposite of the instruction.

3. **Benchmark.** Ask "What was revenue in Q1?" again. Then add the question as a benchmark whose ground truth answers it in the **same shape**, a single total:

```text
SELECT sum(amount) AS revenue
FROM workspace.default.genie_orders
WHERE order_date >= '2026-02-01' AND order_date < '2026-05-01'
```

Run the benchmark. The judge compares Genie's result with the ground truth's. A ground truth grouped by calendar quarter returns 420 and 80 and marks Genie's correct 500 as wrong, and a benchmark with no ground truth SQL only shows "Needs review".

Expected evidence:

```text
before      -> Q1 interpreted as January to March: revenue 920 (checked on Free Edition)
after       -> Q1 interpreted as February to April: revenue 500 (checked on Free Edition)
example SQL -> FY2025 Q4 500 (the January order) ; FY2026 Q1 500 ; FY2026 Q2 250
after the example -> Genie reuses its pattern: YEAR and QUARTER of ADD_MONTHS(order_date, -1), fiscal Q1 of 2026, revenue 500
benchmark   -> 100% accurate (1/1), assessment Good, with the single-total ground truth (checked on Free Edition)
```

**Cleanup:** delete the agent, then drop `workspace.default.genie_orders` and `workspace.default.genie_customers`.